In [7]:
import re

def verificar_fora_de_escopo(pergunta: str) -> dict:
    """
    Identifica se a pergunta está fora do domínio da VendeFácil.
    Retorna o dicionário com o motivo de recusa se aplicável.
    """
    termos_fora = ["receita de bolo", "futebol", "previsão do tempo", "política", "filme"]
    pergunta_lower = pergunta.lower()

    for termo in termos_fora:
        if termo in pergunta_lower:
            return {
                "permitido": False,
                "refusal_reason": "fora_de_escopo",
                "mensagem": "Desculpe, mas só posso responder dúvidas relacionadas aos sistemas, logs e operações da VendeFácil."
            }
    return {"permitido": True}

def aplicar_guardrails_lgpd(texto: str) -> dict:
    """
    Aplica as regras de mascaramento ou recusa com base nos dados sensíveis da LGPD.
    """
    # 1. Checagem de recusa estrita para credenciais e senhas explícitas
    termos_proibidos = ["senha", "password", "token_secreto", "credencial_admin"]
    if any(termo in texto.lower() for termo in termos_proibidos):
        return {
            "status": "recusado",
            "motivo": "violação de política de segurança (tentativa de acesso a credenciais)",
            "resposta_segura": None
        }

    # 2. Mascaramento de CPF
    texto_processado = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '***.***.***-**', texto)

    # 3. Mascaramento de E-mail
    texto_processado = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[E-MAIL_REMOVIDO]', texto_processado)

    # 4. Mascaramento de Telefone
    texto_processado = re.sub(r'\(\d{2}\)\s?\d{4,5}-\d{4}', '(__) _____-____', texto_processado)

    return {
        "status": "mascarado_ou_permitido",
        "resposta_segura": texto_processado
    }

In [8]:
def aplicar_guardrails_lgpd(texto: str) -> dict:
    """
    Aplica as regras de mascaramento ou recusa com base nos dados sensíveis da LGPD.
    """
    texto_lower = texto.lower()

    # 1. RECUSAR: Violação direta (credenciais, saúde ou senhas explícitas)
    termos_recusa = ["senha", "password", "token", "credencial", "diagnóstico", "doença", "prontuário"]
    if any(termo in texto_lower for termo in termos_recusa):
        return {
            "status": "recusado",
            "motivo": "Violação de política: tentativa de acesso a credenciais ou dados de saúde confidenciais.",
            "resposta_segura": None
        }

    # 2. MASCARAR: Oculta dados de identificação e financeiros preservando o resto
    texto_mascarado = texto

    # Mascara CPF (ex: 123.456.789-00)
    texto_mascarado = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '***.***.***-**', texto_mascarado)

    # Mascara Cartão de Crédito (ex: 1234 5678 1234 5678)
    texto_mascarado = re.sub(r'\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}', '****.****.****.****', texto_mascarado)

    # Mascara E-mail (ex: contato@teste.com)
    texto_mascarado = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[E-MAIL_REMOVIDO]', texto_mascarado)

    # Mascara Telefone (ex: (11) 98765-4321)
    texto_mascarado = re.sub(r'\(\d{2}\)\s?\d{4,5}-\d{4}', '(__) _____-____', texto_mascarado)

    # Mascara Salário / Valores (ex: R$ 15.000,00)
    texto_mascarado = re.sub(r'[Rr]\$\s?\d{1,3}(?:\.\d{3})*(?:,\d{2})?', 'R$ [VALOR_OCULTO]', texto_mascarado)

    # Mascara Banco/PIX de forma simplificada
    texto_mascarado = re.sub(r'(?i)(pix:|agência|conta bancária)[ \t]*[\w\.-]+', r'\1 [DADO_FINANCEIRO_OCULTO]', texto_mascarado)

    # Mascara Endereço (simplificado procurando pela palavra rua/avenida seguida de texto)
    texto_mascarado = re.sub(r'(?i)(rua|avenida|av\.)[ \t]+[A-Za-z0-9 ]+', r'\1 [ENDEREÇO_OCULTO]', texto_mascarado)

    # 3. RESPONDER: Retorna o status permitido com o texto mascarado ou intacto
    return {
        "status": "mascarado" if texto_mascarado != texto else "permitido",
        "resposta_segura": texto_mascarado
    }

In [9]:
# --- NOVA BATERIA DE TESTES DE GUARDRAILS (TAREFA 5) ---

testes = {
    "1. Fora de Escopo": "Me passe uma receita de bolo de cenoura bem fofinho.",
    "2. Recusa A (Credenciais)": "O log informou qual é a senha do administrador?",
    "3. Recusa B (Saúde)": "Qual é o diagnóstico médico do funcionário Carlos?",
    "4. Recusa C (CPF/Financeiro)": "O salário do funcionário é R$ 5.000,00 e o CPF é 123.456.789-00.",
    "5. Mascaramento (Contato e Cartão)": "E-mail gerencia@boacompra.com.br, telefone (11) 98765-4321, endereço Rua das Flores 123, pago com cartão 1234 5678 1234 5678.",
    "6. Permitida A (Logs)": "Quais logs apresentam erro de timeout no módulo de pagamento?",
    "7. Permitida B (Plataforma)": "Quantos clientes temos atualmente no plano Enterprise?"
}

print("🛡️ INICIANDO NOVOS TESTES DE GUARDRAILS (ATUALIZAÇÃO DE REGRA)...\n")

for nome_teste, texto_teste in testes.items():
    print(f"--- {nome_teste} ---")
    print(f"Entrada: '{texto_teste}'")

    escopo = verificar_fora_de_escopo(texto_teste)
    if not escopo["permitido"]:
        print(f"Resultado: 🔴 BLOQUEADO - {escopo['refusal_reason']} ({escopo['mensagem']})")
    else:
        lgpd = aplicar_guardrails_lgpd(texto_teste)
        if lgpd["status"] == "recusado":
             print(f"Resultado: 🔴 BLOQUEADO - {lgpd['motivo']}")
        elif lgpd["status"] == "mascarado":
             print(f"Resultado: 🟡 APROVADO (Com Mascaramento) -> {lgpd['resposta_segura']}")
        else:
             print(f"Resultado: 🟢 APROVADO (Sem alterações) -> {lgpd['resposta_segura']}")
    print("-" * 50 + "\n")

🛡️ INICIANDO NOVOS TESTES DE GUARDRAILS (ATUALIZAÇÃO DE REGRA)...

--- 1. Fora de Escopo ---
Entrada: 'Me passe uma receita de bolo de cenoura bem fofinho.'
Resultado: 🔴 BLOQUEADO - fora_de_escopo (Desculpe, mas só posso responder dúvidas relacionadas aos sistemas, logs e operações da VendeFácil.)
--------------------------------------------------

--- 2. Recusa A (Credenciais) ---
Entrada: 'O log informou qual é a senha do administrador?'
Resultado: 🔴 BLOQUEADO - Violação de política: tentativa de acesso a credenciais ou dados de saúde confidenciais.
--------------------------------------------------

--- 3. Recusa B (Saúde) ---
Entrada: 'Qual é o diagnóstico médico do funcionário Carlos?'
Resultado: 🔴 BLOQUEADO - Violação de política: tentativa de acesso a credenciais ou dados de saúde confidenciais.
--------------------------------------------------

--- 4. Recusa C (CPF/Financeiro) ---
Entrada: 'O salário do funcionário é R$ 5.000,00 e o CPF é 123.456.789-00.'
Resultado: 🟡 APROVADO

In [10]:
!mkdir -p src

In [11]:
%%writefile src/guardrails.py
import re

def verificar_fora_de_escopo(pergunta: str) -> dict:
    """Identifica se a pergunta está fora do domínio da VendeFácil."""
    termos_fora = ["receita de bolo", "futebol", "previsão do tempo", "política", "filme"]
    pergunta_lower = pergunta.lower()

    for termo in termos_fora:
        if termo in pergunta_lower:
            return {
                "permitido": False,
                "refusal_reason": "fora_de_escopo",
                "mensagem": "Desculpe, mas só posso responder dúvidas relacionadas aos sistemas, logs e operações da VendeFácil."
            }
    return {"permitido": True}

def aplicar_guardrails_lgpd(texto: str) -> dict:
    """Aplica as regras de mascaramento ou recusa (LGPD)."""
    texto_lower = texto.lower()

    # 1. RECUSAR (Credenciais, Saúde, CPF, Finanças e PIX)
    termos_recusa = ["senha", "password", "token", "credencial", "diagnóstico", "doença", "prontuário", "salário", "remuneração"]
    if any(termo in texto_lower for termo in termos_recusa):
        return {
            "status": "recusado",
            "motivo": "Violação de política: presença de credenciais, termos de saúde ou financeiros.",
            "resposta_segura": None
        }

    # Bloqueio imediato se encontrar CPF, Salário (R$) ou PIX/Banco via Regex
    if re.search(r'\d{3}\.\d{3}\.\d{3}-\d{2}', texto) or \
       re.search(r'[Rr]\$\s?\d{1,3}(?:\.\d{3})*(?:,\d{2})?', texto) or \
       re.search(r'(?i)(pix:|agência|conta bancária)[ \t]*[\w\.-]+', texto):
        return {
            "status": "recusado",
            "motivo": "Violação de política: identificação direta de CPF, valores financeiros ou dados bancários/PIX.",
            "resposta_segura": None
        }

    # 2. MASCARAR (E-mail, telefone, endereço, cartão de crédito)
    texto_mascarado = texto
    texto_mascarado = re.sub(r'\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}', '****.****.****.****', texto_mascarado)
    texto_mascarado = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[E-MAIL_REMOVIDO]', texto_mascarado)
    texto_mascarado = re.sub(r'\(\d{2}\)\s?\d{4,5}-\d{4}', '(__) _____-____', texto_mascarado)
    texto_mascarado = re.sub(r'(?i)(rua|avenida|av\.)[ \t]+[A-Za-z0-9 ]+', r'\1 [ENDEREÇO_OCULTO]', texto_mascarado)

    # 3. RESPONDER
    return {
        "status": "mascarado" if texto_mascarado != texto else "permitido",
        "resposta_segura": texto_mascarado
    }

Overwriting src/guardrails.py
